# 08 · 用 MLflow 原生的方式做评测：数据集、打分器、追踪

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → [08 MLflow 评测 ✅] → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 起：改进 agent …
```

> **这一章的代码在合并请求 #8 里。** 第 3 步会真的跑两个对照系统，每跑一次就在 MLflow 里多一条记录，可以重复运行。

## 背景：评测框架，公司里不自己写

第 07 章的第一版里，"逐题去问、逐题判分、存结果"全是我们自己写的：一个循环，加一个自建的文件夹。后来把"存"换成了 MLflow，但"问"和"判"还是自己写的。

MLflow 自带的使用说明里有一句话写得很重：**不要自己写评测框架**，要用它原生的数据集、打分器和 `mlflow.genai.evaluate()`。原因是：用原生的做法，题目、每道题的回答、每道题的评分、汇总分数都记在同一个地方，而且互相连得上；自己写的循环会漏掉这些联系。Databricks 上做大模型应用的评测，用的也是这一套。

**判分规则（第 07 章那四条）一条都不变**，只是换成 MLflow 的标准接口来跑。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 题集登记成 MLflow 评测数据集 | `mlflow.genai.datasets` | 题目和标准答案在 MLflow 里也有一份，每次评测都连得上 |
| 2 | 判分规则写成代码打分器 | `@scorer`（代码检查） | 规则不变，接口是 MLflow 标准的 |
| 3 | 用 MLflow 跑评测 | `mlflow.genai.evaluate()` | 不用自己写循环；自动并行 |
| 4 | 每道题一条追踪记录 | MLflow Tracing（`@mlflow.trace`） | 每道题问了什么、答了什么、错在哪，都留下来 |
| 5 | 实验记录 + 完整性检查 | MLflow 运行 + 核对记录数 | 分数可比较、可重算；记录不全就报错 |

**花费：0。** 还不调大模型。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 题集登记成 MLflow 的评测数据集

### 在干什么
把第 06 章冻结的题号清单，在 MLflow 里登记成一份"评测数据集"。每条记录有两部分：**题目**（inputs）和**标准答案**（expectations）。

### 背景
题号清单在 git 里，是"考哪些题"的定义，这一点不变。但 MLflow 里的每次评测，要能指回"用的是哪份题"；在网页上，也要能直接看到每道题和它的标准答案。

### 用了什么技术，为什么
`mlflow.genai.datasets`：MLflow 的评测数据集功能。

| 部分 | 里面有什么 | 谁看得到 |
|---|---|---|
| inputs（题目） | 病例编号、症状、性别 | 被测的系统 |
| expectations（标准答案） | OMIM 编号、病名、致病基因 | 只有打分器 |

数据集用清单的名字命名（`dev-200`），并记下清单的指纹。指纹对不上就拒绝使用：清单是冻结的，登记的副本也不能悄悄变。

### 公司里什么情况怎么做
- **评测数据集在 MLflow 里集中管理**，团队共用，网页上就能浏览
- **换题**：新题集是一个新数据集，旧的不改。以前的评测仍然连着以前的题

### 执行
取出 dev-200 的数据集。如果还没登记过，这一步会先登记（只有第一次）：

In [ ]:
import json
from raredx.config import settings
from raredx.data import cases as case_store, sources
from raredx.evals import splits
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
source = sources.load_sources(ROOT / "evals/datasets.toml")["phenopacket-store"]
by_id = {c.id: c for c in case_store.read(sources.processed_dir(source, ROOT / "data"))}
manifest = ROOT / "evals/splits/dev-200.tsv"
cases = [by_id[case_id] for case_id, _ in splits.read(manifest)]

dataset = harness.dataset("dev-200", sources.sha256_of(manifest), cases)
records = dataset.to_df()
print(f"数据集 {dataset.name}：{len(records)} 条，登记时的清单指纹 {dataset.tags['manifest_sha256'][:12]}…\n")
print("一条记录的题目（inputs）：")
print(json.dumps(records.iloc[0]["inputs"], ensure_ascii=False)[:300], "…")
print("\n同一条记录的标准答案（expectations）：")
print(json.dumps(records.iloc[0]["expectations"], ensure_ascii=False))

### 结果怎么看
- 200 条记录，清单指纹和第 06 章 dev-200 清单的一致
- 题目里只有症状；病名、编号、基因都在标准答案里。系统只会拿到题目

### 面试怎么说
> "Each frozen manifest is registered as an MLflow evaluation dataset — inputs for what the system sees, expectations for what the scorer checks — tagged with the manifest's checksum."

---

## 第 2 步 · 判分规则写成"代码打分器"

### 在干什么
把第 07 章的判分规则，包成 MLflow 的**打分器**（scorer）。规则本身不变。

### 背景
MLflow 的打分器有三种：**代码检查**、**内置检查**、**大模型当裁判**。MLflow 自己的建议是：能用代码判断对错的，就用代码检查，因为最便宜、最稳定。我们按编号判，正好是这种。

### 用了什么技术，为什么
一个加了 `@scorer` 的函数（在 [`src/raredx/evals/harness.py`](../../src/raredx/evals/harness.py)）：收到系统的回答和标准答案，返回一组评分：

| 评分 | 意思 |
|---|---|
| `answered` | 交了答卷没有（出错、没回答就是 False） |
| `disease_top1` … `disease_top10` | 正确的病在不在前 1/3/5/10 名 |
| `gene_top1` … `gene_top10` | 正确的基因在不在前 1/3/5/10 名（病例没有致病基因时，这几项不出现） |

MLflow 会自动算出每一项在所有题上的平均值，也就是命中率。

### 公司里什么情况怎么做
- **大模型当裁判**只用在没法用代码判的地方，比如"解释写得好不好"，而且要先抽一批人工核对，确认裁判判得准
- **改判分规则**：改代码、改测试、走合并请求；旧的评测用新规则重判（第 5 步）

### 执行
用第 07 章那份假想的答卷（第 1 名猜 Perrault 1 型，第 2 名猜 7 型）试一下打分器：

In [ ]:
from raredx.evals.harness import diagnosis

answer = {"candidates": [
    {"disease_name": "Perrault syndrome 1", "disease_id": "OMIM:233400", "gene": "HSD17B4"},
    {"disease_name": "Perrault syndrome 7", "disease_id": "621101", "gene": "DAP3"},
]}
for verdict in diagnosis(outputs=answer, expectations={"disease_id": "OMIM:621101", "gene": "DAP3"}):
    print(f"{verdict.name:<14} {verdict.value}")

### 结果怎么看
- `disease_top1 False`，`disease_top3 True` 及以后都是 True：正确答案排在第 2 名，和第 07 章判的一样
- 基因也一样：第 2 名才对

### 面试怎么说
> "The scoring rules are a code-based MLflow scorer: deterministic, free, and returning one feedback per check, so MLflow aggregates top-k accuracy directly."

---

## 第 3 步 · 用 mlflow.genai.evaluate 跑评测

### 在干什么
把"系统、数据集、打分器"交给 `mlflow.genai.evaluate()`。它负责逐题去问系统、逐题打分、最后汇总。

### 背景
原来这一步是我们自己写的循环。交给 MLflow 以后，它还会**自动并行**：同时问 10 道题。下一章调大模型时，这能快上好几倍。

### 用了什么技术，为什么
`raredx eval run` 现在做的事：

```
取出（或登记）数据集
  → 开一条 MLflow 运行，记下参数（哪个系统、哪份题、哪份数据、哪版代码）
  → mlflow.genai.evaluate(数据集, 系统, 打分器)：并行地逐题问、逐题打分
  → 核对：存下来的记录数 = 题数（第 5 步）
  → 记下分数和误差范围
```

> **一个真实踩过的坑：线程锁死。** 第一次在 200 道题上跑，程序卡住了：不报错，也不占 CPU，就是不动。查下来是 MLflow 在每个并行线程里，同时去检查电脑上有没有装一个叫 litellm 的包。我们没装它，而 Python 3.12 在多个线程同时尝试导入同一个不存在的包时，会互相锁死：10 道题时没事，40 道题时所有线程都卡住了。办法是在评测开始前先告诉 Python"这个包不存在"，这个检查就会立刻返回。代码里有注释说明原因。

### 公司里什么情况怎么做
- **并行度要按接口限流来调**：调大模型时，同时发太多请求会被服务商拒绝
- **程序不动的时候**：先看 CPU。占满是在算，接近 0 是在等，多半是卡住了，这时要看每个线程停在哪一行

### 执行

In [ ]:
run("uv run raredx eval run --system oracle --split dev-200")
run("uv run raredx eval run --system popularity --split dev-200")

### 结果怎么看
- 先是 MLflow 自己的几行提示（`Evaluation completed`，以及"去网页的 Traces 页看每道题"）
- 然后是我们的分数表：oracle 每一项 100%；popularity 前 10 名 9.0%。**和第 07 章自己写的循环算出来的一模一样**：换成 MLflow 原生的做法，规则和结果都没变
- `200 cases, 200 answered`：200 道题都有回答

### 面试怎么说
> "Evaluation runs through mlflow.genai.evaluate: dataset, predict function and scorers in, parallel execution, per-row traces and aggregated metrics out — no hand-written evaluation loop."

---

## 第 4 步 · 每道题一条追踪记录

### 在干什么
每问一道题，MLflow 都记一条**追踪**（trace）：题目、回答、用时、有没有出错（出错的话连错误原因一起），以及打分器给这道题的评分。

### 背景
只看总分，不知道错在哪里。公司里排查大模型应用的问题，第一步就是翻追踪记录。下一章调大模型时，追踪里还会多出一层：模型收到的完整提示词、它的回答、用了多少 token、花了多少时间。

### 用了什么技术，为什么
每道题的入口函数加了 `@mlflow.trace`，开一条追踪记录，根节点就是系统的名字。系统内部调用的模型、工具，会作为下一层节点挂在下面。

| 追踪记录里有 | 例子 |
|---|---|
| 题目和回答 | 症状列表；前 10 名候选 |
| 状态和用时 | OK 或 ERROR；多少毫秒 |
| 评分 | 这道题的 `disease_top1` … `gene_top10` |

### 公司里什么情况怎么做
- **线上服务也一直记追踪**：用户反馈"答错了"，按时间、按用户就能找到那一次的完整记录
- 同一套概念在 Databricks、LangSmith、Langfuse 里都一样，换工具不用重学

### 执行
取出刚才 popularity 那次评测里的一条追踪记录：

In [ ]:
import mlflow

popularity = next(r for r in harness.runs("dev-200") if r["name"] == "popularity")
experiment_id = harness.client.get_experiment_by_name("eval/dev-200").experiment_id
[trace] = mlflow.search_traces(locations=[experiment_id], run_id=popularity["run_id"],
                               return_type="list", max_results=1)
root = trace.data.spans[0]
print("追踪记录：", trace.info.trace_id, "| 状态：", trace.info.state.value)
print("根节点：  ", root.name)
print("题目：    ", root.inputs["question"]["case_id"], root.inputs["question"]["hpo_labels"][:3], "…")
print("回答前 3：", [c["disease_name"] for c in root.outputs["candidates"][:3]])
print("评分：    ", {a.name: a.value for a in trace.info.assessments if a.name.startswith(("answered", "disease_top"))})

### 结果怎么看
- 状态 `OK`；根节点是 `popularity`
- 题目是这个病人的症状；回答的前 3 名，永远是那几个"最常见的病"（瞎猜系统不看症状）
- 评分：这道题 `answered` 是 True，`disease_top1` 到 `disease_top10` 多半都是 False：正确的病不在那 10 个里

### 面试怎么说
> "Every question is a trace — inputs, outputs, latency, status and the scorer's verdicts — so a wrong answer can be inspected directly; model and tool calls nest underneath as child spans."

---

## 第 5 步 · 实验记录、完整性检查、在网页上看

### 在干什么
每次评测是一条 MLflow **运行**（run）：参数、分数（MLflow 算的平均值，加上我们算的误差范围）、所有追踪记录。评测完，先核对记录有没有存全。

### 背景
第 07 章开头说的教训：旧项目一份报告写着 70%，重跑是 83–85%，因为当时没记下用的是哪版代码、哪份数据。

> **另一个真实踩过的坑：记录被悄悄丢掉。** MLflow 用本机数据库文件时，默认最多同时开 15 个连接。评测时有 20 多个线程同时往里写（答题、打分、写追踪），有的写入等了 30 秒拿不到连接，MLflow 就把那条记录丢掉，只打一行警告。第一次跑瞎猜系统，200 道题只存下了 150 条，汇总表里只数出 132 道，分数也跟着错了。
>
> 办法两条：**连接池调大到 40**，让写入排队等，而不是超时丢掉；**每次评测完核对"存下来的记录数 = 题数"**，不等就报错，这次运行标为失败，不报一个错的分数。

### 用了什么技术，为什么

| MLflow 里的词 | 是什么 | 在我们这里 |
|---|---|---|
| 实验（experiment） | 一组可以互相比较的记录 | 每份题集一个，比如 `eval/dev-200` |
| 运行（run） | 一次评测 | 每跑一次 `raredx eval run` 就多一条 |
| 参数（params） | 怎么跑的 | 系统、题集和它的指纹、数据、代码版本 |
| 指标（metrics） | 结果 | 每项命中率（`/mean`）和误差范围（`/low`、`/high`） |
| 追踪记录（traces） | 每道题的完整记录 | 第 4 步 |
| 数据集（dataset） | 题目和标准答案 | 第 1 步 |

- **本机模式**：记录存在项目里的 `mlflow.db`，文件存在 `mlruns/`，两样都不进 git；不用注册账号，断网也能用。换成公司的 MLflow 服务器，**只改一个地址**
- **代码版本带 `-dirty`**：跑的时候代码、题集或依赖有没提交的改动，就加上这个标记；这样的结果不能写进报告
- **重新判分不重跑**：`raredx eval score <运行编号>` 把那次评测存下来的追踪记录交给打分器再判一遍，记成一条新的运行，**不再问系统**。对大模型来说，就是不用再花一次钱

### 公司里什么情况怎么做
- **团队共用一台 MLflow 服务器**，所有人的评测都在一起
- **比较两个版本**：网页上勾选两条运行，点 **Compare**
- **正式报告只用干净代码跑出来的结果**（没有 `-dirty`）
- **同类工具**：Weights & Biases、LangSmith，都要注册账号，记录存在别人的服务器上

### 执行
列出 dev-200 上的所有运行；再把最新那次 popularity 不重跑、重新判一次分：

In [ ]:
run("uv run raredx eval runs --split dev-200")
latest = next(r["run_id"] for r in harness.runs("dev-200") if r["name"] == "popularity")
run(f"uv run raredx eval score {latest}")

### 结果怎么看
- `eval runs`：每行一条运行，最新的在最上面，能看到 oracle（100%）和 popularity（前 10 名 9.0%）
- 重新判分：数字和第 3 步一模一样，最后一行是新运行的编号。这次没有再问系统，只是把存下来的追踪记录重判了一遍

### 在浏览器里看
下面这一格在后台打开 MLflow 的网页：

In [ ]:
import socket, subprocess, time

with socket.socket() as probe:
    already_open = probe.connect_ex(("127.0.0.1", 5050)) == 0
if already_open:
    ui = None
    print("5050 端口上已经有一个 MLflow 网页在运行：http://127.0.0.1:5050")
else:
    ui = subprocess.Popen(["uv", "run", "mlflow", "ui", "--port", "5050"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    time.sleep(8)
    print("MLflow 网页已打开：http://127.0.0.1:5050  （看完运行下一格关掉它）")

点开 http://127.0.0.1:5050 ，然后：

1. 左边点实验 **eval/dev-200**：每一行是一次评测，能看到运行名、时间、各项分数
2. 点开 `popularity` 那一行：
   - **Parameters / Metrics**：怎么跑的、分数
   - **Traces**：200 道题，每道一条；点开一条，能看到题目、回答、每项评分
3. 回到实验页面，看 **Datasets**：登记的 `dev-200` 数据集
4. 回到列表，勾选 `oracle` 和 `popularity` 两行，点 **Compare**：参数和分数并排显示

> 端口用 5050 而不是 MLflow 默认的 5000：苹果电脑的"隔空播放接收器"占着 5000。

看完运行这一格，关掉网页服务：

In [ ]:
import os, signal

if ui is not None:
    try:
        os.killpg(ui.pid, signal.SIGTERM)   # 关掉网页服务，连同它启动的子进程
    except ProcessLookupError:
        pass                                # 已经自己退出了
print("已关闭")

### 面试怎么说
> "Every evaluation is an MLflow run with its params, metrics and per-question traces. After each run I check that every answer was recorded — a run with missing records fails instead of reporting a number."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 数据集 | `mlflow.genai.datasets` | 题目和标准答案登记在 MLflow，评测连得上题 |
| 2 打分器 | `@scorer` 代码检查 | 规则确定、免费；MLflow 直接汇总 |
| 3 评测 | `mlflow.genai.evaluate()` | 不自己写循环；自动并行 |
| 4 追踪 | `@mlflow.trace` | 每道题问了什么、答了什么都看得到 |
| 5 运行 + 核对 | MLflow 运行 + 记录数检查 | 可比较、可重算；记录不全就不报分 |

**一句话**：评测从"自己写的循环"换成了 MLflow 原生的一整套：数据集、打分器、追踪、运行。规则和结果都没变，但每道题都看得到，也接得上公司的服务器。下一章，真正的大模型第一次上场。

## 面试题

1. **How do you evaluate an LLM application with MLflow?**
   > "Register the test cases as an evaluation dataset, write the checks as scorers — code checks where the answer can be verified mechanically, judges where it can't — and run mlflow.genai.evaluate with the app's predict function. Every row becomes a trace with its assessments, and the run holds the aggregated metrics."

2. **Why did you choose a code scorer over an LLM judge?**
   > "The answer carries an OMIM identifier and a gene symbol, so correctness is a deterministic comparison. A code check is free, exact and stable; a judge would add cost and noise for no gain."

3. **How do you make sure evaluation numbers are trustworthy?**
   > "Controls — an oracle that must score 100% and a floor baseline — failures counted as misses, and a completeness check: if fewer answers were recorded than cases asked, the run fails rather than reporting a number. I found that the hard way: SQLite connection-pool timeouts were silently dropping traces."

4. **What is tracing, and why does it matter for agents?**
   > "A trace records one request end to end — inputs, outputs, latency, errors, and every nested model or tool call. For an agent, it's how you see which tool it chose and why, and how you debug a wrong answer."

**下一章 09 · 第一个基线：大模型直接猜**：第一次调大模型。把症状交给它，一次回答；每一次调用都记进 MLflow 的追踪里。